In [43]:
import os
import re
import csv
import json
import pickle
import random
import torch
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from copy import deepcopy
from typing import Union, Tuple, Dict
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [44]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        # lines = f.readline()
        # lines = json.loads(lines)
        # for line in lines:
        #     data.append(line)
        
        for line in f:
            data.append(json.loads(line))
    return data

In [45]:
data = jsonl_load('../data/QA_Datasets/ambigqa/ambignq_with_evidence_articles/dev_with_evidence_articles.json')
data = data[0]
len(data)

2002

In [51]:
qa_list = []
cache = {}
idx = 0
contents2docid = {}

for sample in data:
    qid = sample['id']
    question = sample['question']
    answer = []
    for ans in sample['annotations']:
        if 'qaPairs' in ans:
            qa_pairs = ans['qaPairs']
            for qa_pair in qa_pairs:
                answer.extend(qa_pair['answer'])
        else:
            answer.extend(ans['answer'])
    answer = [str(ans) for ans in answer]

    docs = sample['articles_plain_text']
    new_docs = []
    for doc in docs:
        if len(doc) == 0:
            contiㅇnue
        docid = contents2docid[doc] if doc in contents2docid else idx
        if docid == idx:
            idx += 1
        new_docs.append({'id':docid, 'contents':doc})
        
    if len(new_docs) == 0:
        continue

    cache[question] = new_docs
    
    qa_list.append({
        'id': qid, 
        'Question': question, 
        'answer': answer, 
    })

len(qa_list), len(cache)

(2002, 2002)

In [53]:
path = '/data/Search-o1/data/QA_Datasets/ambigqa'
json_dump(os.path.join(path, 'ambigqa_full.json'), qa_list)

path = '/data/Search-o1/cache/ambigqa'
json_dump(os.path.join(path, 'search_cache_full.json'), cache)

In [54]:
random.seed(42)
random.shuffle(qa_list)
sampled_qa_list = qa_list[:500]
sampled_cache = {}
for qa in sampled_qa_list:
    question = qa['Question']
    sampled_cache[question] = deepcopy(cache[question])
len(sampled_qa_list), len(sampled_cache)

(500, 500)

In [56]:
path = '/data/Search-o1/data/QA_Datasets/ambigqa'
json_dump(os.path.join(path, 'ambigqa_500.json'), sampled_qa_list)

path = '/data/Search-o1/cache/ambigqa'
json_dump(os.path.join(path, 'search_cache_500.json'), sampled_cache)